# 05 — Production RAG Operations

**Track:** Advanced · **Stage:** Capstone

A production system is not just one giant LLM prompt. It is an offline knowledge pipeline plus an online answer pipeline, surrounded by evaluation, authorization, observability, and rollback controls.

In this final deep dive, we simulate **Observability** (Latency, Token Cost, Tracing) and **Fallback handling**, which are critical for Day-2 RAG operations.

1. **Part 1: The Theory of Production RAG.** We will define Service Level Objectives (SLOs), Error Budgets, and how to calculate the true *Cost per Grounded Answer*.
2. **Part 2: Production Observability with LangChain.** We will build a custom callback tracer to monitor latency and simulate token costs across a mock production pipeline.

---
## Part 1: The Theory of Production RAG

Before writing code, a production engineer must define the bounds of the system. 

### Service Level Objectives (SLOs) & Error Budgets
An SLO defines the target health of your service. An Error Budget is the allowed threshold of failure before you must freeze feature deployments and fix reliability.
- **Latency SLO:** p95 < 3s, p99 < 8s.
- **Availability SLO:** 99.9% uptime (allows ~43 minutes of downtime a month).
- **Quality SLO:** Citation Faithfulness > 95% on supported queries.

### Cost per Grounded Answer
RAG systems have multiple cost centers (Embedding inference, Vector Search infra, Reranker inference, LLM inference). The most critical metric is not just "Cost per Query", but the **Cost per Grounded Answer**:

```python
cost_per_success = (
    total_cost_usd
    / (n_queries * faithfulness_rate * citation_valid_rate * grounded_rate)
)
```
If a cheaper model saves 20% on inference but hallucinates 40% more often, the *Cost per Grounded Answer* actually went up. Always optimize for groundedness first.

---
## Part 2: Production Observability with LangChain

In a real production environment, you would use tools like **LangSmith**, **Phoenix (Arize)**, or **Datadog** to trace calls.

Here, we simulate this by hooking into LangChain's callback system to manually trace our request trajectory, measure stage-by-stage latency, and simulate token cost counting.

In [ ]:
# !pip install langchain langchain-core

import time
from typing import Any, Dict, List
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms.fake import FakeListLLM
from langchain_core.runnables import RunnablePassthrough

### Step A: Build a Custom Callback Handler

We create a callback handler that hooks into the start and end of every LLM and Retriever call. This allows us to track latency and simulate token costs.

In [ ]:
class ProductionMetricsCallback(BaseCallbackHandler):
    def __init__(self):
        self.start_times = {}
        self.metrics = {"llm_calls": 0, "total_latency_ms": 0.0, "estimated_cost": 0.0}

    def on_llm_start(self, serialized: Dict[str, Any], prompts: List[str], **kwargs: Any) -> Any:
        self.start_times[kwargs["run_id"]] = time.time()
        self.metrics["llm_calls"] += 1
        # Simulate token counting (naive length approximation)
        prompt_len = sum(len(p) for p in prompts)
        self.metrics["estimated_cost"] += (prompt_len / 1000) * 0.0015 # Simulate $0.0015 per 1K input tokens

    def on_llm_end(self, response, **kwargs: Any) -> Any:
        run_id = kwargs["run_id"]
        if run_id in self.start_times:
            elapsed_ms = (time.time() - self.start_times[run_id]) * 1000
            self.metrics["total_latency_ms"] += elapsed_ms
            
            gen_len = sum(len(g.text) for gen_list in response.generations for g in gen_list)
            self.metrics["estimated_cost"] += (gen_len / 1000) * 0.0020 # Simulate $0.0020 per 1K output tokens

    def print_metrics(self):
        print("\n--- [PROD TRACE METRICS] ---")
        print(f"LLM Calls:      {self.metrics['llm_calls']}")
        print(f"Total Latency:  {self.metrics['total_latency_ms']:.2f} ms")
        print(f"Estimated Cost: ${self.metrics['estimated_cost']:.6f}")
        print("----------------------------")

### Step B: Build the Mock Production Pipeline

We assemble a standard pipeline: retrieve -> format prompt -> LLM -> parse output.

In [ ]:
# 1. Mock Retriever 
def mock_retrieve(query: str) -> str:
    # Simulate a fast database lookup
    time.sleep(0.05) 
    return "Acme internal deployment guidelines: All changes require 2 approvals."

# 2. Mock LLM (simulating a 200ms generation delay)
class SlowFakeLLM(FakeListLLM):
    def _call(self, prompt: str, stop=None, run_manager=None, **kwargs) -> str:
        time.sleep(0.2)
        return super()._call(prompt, stop, run_manager, **kwargs)

llm = SlowFakeLLM(responses=["Based on the provided guidelines, you need 2 approvals."])

# 3. Prompt
prompt = ChatPromptTemplate.from_template(
    "Answer the user based on the context.\nContext: {context}\nUser: {question}"
)

# 4. LCEL Chain
chain = (
    {"context": mock_retrieve, "question": RunnablePassthrough()} 
    | prompt 
    | llm 
    | StrOutputParser()
)

### Step C: Execute with Observability

Watch how the metrics are captured dynamically as the chain runs.

In [ ]:
tracer = ProductionMetricsCallback()

print("Processing user request in production...")
answer = chain.invoke("How many approvals do I need?", config={"callbacks": [tracer]})

print(f"\nFinal Answer: {answer}")
tracer.print_metrics()

## Congratulations!

You have completed the **Advanced RAG Curriculum**. 

By passing this capstone, you now know that designing a RAG system is far more than just stuffing a vector search into a prompt. 
You know how to:
- **Evaluate** and correct bad retrievals (Corrective RAG)
- **Graph** complex multi-hop relationships (GraphRAG)
- **Safeguard** autonomous agents using strict tool boundaries (Agentic RAG)
- **Aggregate** exact math directly using code agents (Structured Data)
- **Route** queries effectively before they hit expensive pipelines (Adaptive RAG)
- **Observe** and constrain your pipelines in a real environment (Production RAG)

You are ready to build enterprise RAG.